In [15]:
import pandas as pd
df = pd.read_csv("data/reference.csv")
print(df.to_string(max_rows=8))

      step    lat     lon     pmin_hpa    vmax_ms    vmax_kt                 init            sid   name basin                valid  bt_lat  bt_lon  bt_wind_kt  bt_pres_hpa  track_err_km  aifs0_vmax   aifs0_pmin  aifs_dvmax  aifs_vmax_runmax  aifs_pmin_runmin  aifs_vmax_prev6  aifs_dvmax_6h  speed_kmh  obs0_wind  obs0_pres  obs_dwind_12h  obs_dwind_24h  age_h  init_bias  persist_vmax_kt   doy_sin   doy_cos  is_ep  aifs_v2  ri      res_v  corr_vmax_kt  erm_vmax_kt  eta_vmax_kt  eta1980_vmax_kt
0        6  13.50 -105.50  1007.638611  12.805787  24.892452  2025-05-29 00:00:00  2025149N12256  ALVIN    EP  2025-05-29 06:00:00    13.0  -105.6        30.0       1005.0     56.641194   25.391666  1006.245178   -0.499214         25.391666       1006.245178        25.391666      -0.499214  30.927976       30.0     1006.0            NaN            NaN    6.0  -4.608334        29.500786  0.546711 -0.837321      1        0   0   5.107548     34.157588    41.805071    40.978710        38.958485
1     

In [16]:
RI_ONLY = True

if RI_ONLY:
    df = df[df["ri"] == 1]
    
unique_storms = df["sid"].unique()
print(unique_storms)
df["gbm_abs_err"] = abs(df["bt_wind_kt"] - df["corr_vmax_kt"])
df["eta_abs_err"] = abs(df["bt_wind_kt"] - df["eta_vmax_kt"])

<StringArray>
['2025168N11269', '2025180N13261', '2025207N11219', '2025216N13243',
 '2025223N17337', '2025243N14239', '2025245N17255', '2025261N19312',
 '2025265N14261', '2025268N19306', '2025294N14290', '2026204N08267',
 '2026222N15223', '2026237N09256', '2026238N10223']
Length: 15, dtype: str


In [17]:
import numpy as np

gbm_by_storm = {sid: g.values for sid, g in df.groupby("sid")["gbm_abs_err"]}
eta_by_storm = {sid: g.values for sid, g in df.groupby("sid")["eta_abs_err"]}

gbm_maes, eta_maes, diffs = [], [], []

rng = np.random.default_rng(0)

for _ in range(1000):
    curr_storms = rng.choice(unique_storms, size=len(unique_storms), 
                             replace = True)

    # gather rows of drawn storms
    gbm_errs = np.concatenate([gbm_by_storm[s] for s in curr_storms])
    eta_errs = np.concatenate([eta_by_storm[s] for s in curr_storms])

    gbm_maes.append(gbm_errs.mean())
    eta_maes.append(eta_errs.mean())
    diffs.append(eta_errs.mean() - gbm_errs.mean())

for name, vals in [("GBM", gbm_maes), ("eta", eta_maes), ("eta - GBM", diffs)]:
    lo, hi = np.quantile(vals, [0.025, 0.975])
    print(f"{name}: [{lo:.2f}, {hi:.2f}]")



GBM: [35.86, 45.19]
eta: [24.86, 35.55]
eta - GBM: [-15.32, -6.19]
